# MNIST demo with CREST


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

In [3]:
from crest import DataServer

import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.KerasNode import KerasNode
from crest.model.TensorSpec import TensorSpec


#### Load MNIST Datasets


In [4]:
# Load datasets from server
ds_train, ds_valid, ds_test, ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP, OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

#### Create the HierarchalTensorGraph


In [5]:

layers = [
    KerasNode(Lambda(lambda x: tf.cast(x, tf.float32)), name='layer_0', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28])}),        # Cast dtype
    KerasNode(Lambda(lambda x: tf.expand_dims(x, axis=-1)), name='layer_1', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28, 1])}),  # Add channel dimension
    KerasNode(Conv2D(10, 3, padding='same', activation='relu'), name='layer_2', inputs={INP: TensorSpec(
        [None, 28, 28, 1])}, outputs={OUT: TensorSpec([None, 28, 28, 10])}),  # Convolutional layer
    KerasNode(Flatten(), name='layer_3', inputs={INP: TensorSpec(
        [None, 28, 28, 10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Flatten
    KerasNode(Dropout(0.3), name='layer_4', inputs={INP: TensorSpec(
        [None, 28*28*10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Dropout
    KerasNode(Dense(n_classes, activation='softmax'), name='layer_5', inputs={
              INP: TensorSpec([None, 28*28*10])}, outputs={OUT: TensorSpec([None, 10])})  # Output layer
]

for i in layers[:-1]:
    i.rename_io(outputs_map={OUT: INP})

graph = HierarchalTensorGraph(
    name='MNIST',
    inputs={INP: TensorSpec([None, patch_size, patch_size])},
    outputs={OUT: TensorSpec([None, 10])}
)

graph.add_edge('input', layers[0])

for i, layer in enumerate(layers[1:]):
    graph.add_edge(layers[i], layers[i+1])

graph.add_edge(layers[-1], 'output')

#### Build a model using the HTG


In [6]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)


# Set build options
build_kwargs = {
    'optimizer': 'Adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

/Users/ccruz/projects/terrahydro/development/crest/crest/model/Model.py:82: UserWarning: Use Model.compile instead of Model.build
  warnings.warn('Use Model.compile instead of Model.build')


#### Train the model


In [7]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size': 200,            # 200 samples in each batch
    'features': [[INP], [OUT]],  # [{'image': ...}, {'class': ...}]
    'repeat': True,           # Yield indefinitely
    'workers': 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
        'validation_data': valid,
        'steps_per_epoch': 250,  # 50k samples / 200 samples per batch
        'validation_steps': 50,  # 10k samples / 200 samples per batch
        'epochs': 5,
        'verbose': 1
    })

Epoch 1/5
  1/250 [..............................] - ETA: 43s - loss: 50.3411 - sparse_categorical_accuracy: 0.0750 - custom_metric: 27.6645

2024-02-29 18:15:49.735858: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


250/250 [==============================] - 5s 20ms/step - loss: 2.0617 - sparse_categorical_accuracy: 0.8865 - custom_metric: 27.4092 - val_loss: 0.3388 - val_sparse_categorical_accuracy: 0.9660 - val_custom_metric: 27.4871
Epoch 2/5
250/250 [==============================] - 5s 22ms/step - loss: 0.3272 - sparse_categorical_accuracy: 0.9489 - custom_metric: 27.3751 - val_loss: 0.2270 - val_sparse_categorical_accuracy: 0.9730 - val_custom_metric: 27.4869
Epoch 3/5
250/250 [==============================] - 5s 21ms/step - loss: 0.1947 - sparse_categorical_accuracy: 0.9608 - custom_metric: 27.2691 - val_loss: 0.2071 - val_sparse_categorical_accuracy: 0.9750 - val_custom_metric: 27.4865
Epoch 4/5
250/250 [==============================] - 5s 22ms/step - loss: 0.1322 - sparse_categorical_accuracy: 0.9684 - custom_metric: 27.4218 - val_loss: 0.1613 - val_sparse_categorical_accuracy: 0.9790 - val_custom_metric: 27.4864
Epoch 5/5
250/250 [==============================] - 5s 22ms/step - loss: 

#### Evaluate the model


In [8]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs = {
    'verbose': 1,
    'return_dict': True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - 0s 13ms/step - loss: 0.0833 - sparse_categorical_accuracy: 0.9650 - custom_metric: 26.3262


Accuracy on the full test set should be > ~97%.


### Make predictions


In [9]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size': 200,
    'features': [INP],
    'repeat': True,
    'workers': 0,
    'shuffle': False
})

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size': 200,
    'verbose': False,
    'steps': 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(
        batch_pred, **pred_kwargs)  # Using a batcher
    model_pred_dataset = model.predict(
        ds_pred, **pred_kwargs)   # Using a dataset
    print('Difference:',
          (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions


In [10]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)
x_aux = (lat, lon, x)
ds_aux = StructuredDataset(*x_aux, labels=['lat', 'lon', INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

{'class': array([[4.1070472e-15, 1.0121456e-17, 3.1905685e-13, ..., 9.9999988e-01,
         1.1505534e-12, 4.7267484e-08],
        [1.0132125e-12, 4.5877439e-14, 1.0000000e+00, ..., 8.5967241e-25,
         3.4022996e-14, 2.3842722e-20],
        [4.8853202e-08, 9.9999619e-01, 5.7014694e-07, ..., 6.5818213e-08,
         2.2033707e-06, 3.8187133e-09],
        ...,
        [2.9840435e-17, 2.2821141e-13, 8.1445770e-19, ..., 1.0983618e-08,
         5.3537126e-09, 5.9173232e-07],
        [2.6971177e-14, 4.7556152e-18, 3.0126810e-15, ..., 1.1981905e-11,
         6.2576246e-06, 2.3302917e-17],
        [1.7240181e-17, 6.8475136e-22, 4.8023189e-16, ..., 2.1100840e-21,
         2.5360645e-17, 5.0144231e-18]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [11]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  142.48486804962158 [s]


model save and load

In [12]:
model.save()

In [13]:
loaded_model = model.load('crest_cache')

In [14]:
# check that loaded model performs the same as original
result_loaded = loaded_model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)
result = model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

print('Difference:', (result['class'] - result_loaded['class']).sum())

Difference: 0.0
